# ⚡ FunctionCraft-SLM: 1-Click Cloud Training Pipeline
### End-to-End Distillation, SFT (QLoRA), and DPO Alignment on Free Google Colab / Kaggle

This notebook trains and aligns an enterprise-grade Small Language Model (e.g. **Qwen2.5-1.5B** or **Llama-3.2-1B/3B**) for **ultra-low-latency, zero-hallucination tool calling**.

**Hardware Target:** Free-tier Google Colab T4 GPU (16GB VRAM) or A100.

## 1. Environment & GPU Check

In [ ]:
!nvidia-smi
!pip install -q --upgrade pip
# Install core training libraries with 4-bit bitsandbytes support
!pip install -q torch transformers datasets trl peft bitsandbytes accelerate pydantic pyyaml rich jsonschema

## 2. Clone or Sync FunctionCraft-SLM Repository

In [ ]:
import os
import sys

if not os.path.exists('src'):
    if not os.path.exists('functioncraft-slm'):
        !git clone https://github.com/rb-369/functioncraft-slm.git
    %cd functioncraft-slm

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
print('Workspace ready! Working directory:', os.getcwd())

## 3. Generate Distillation & DPO Datasets

In [ ]:
from src.data.dataset import save_dataset_splits
from src.data.generator import SyntheticDataEngine

print("Generating multi-tool synthetic datasets...")
engine = SyntheticDataEngine(schemas_dir="data/schemas", seed=42)
dataset = engine.generate_dataset(samples_per_tool=250)

saved_files = save_dataset_splits(dataset, output_dir="data/processed")
print("Dataset splits created successfully:")
for split, path in saved_files.items():
    print(f"  - {split}: {len(dataset[split])} records ({path})")

## 4. Stage 1: Supervised Fine-Tuning (SFT) with QLoRA

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
sft_output_dir = "checkpoints/sft_output"

print(f"Loading base model: {base_model_name}")
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

train_ds = load_dataset("json", data_files="data/processed/sft_train.json", split="train")
val_ds = load_dataset("json", data_files="data/processed/sft_val.json", split="train")

training_args = TrainingArguments(
    output_dir=sft_output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    dataset_text_field="messages",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting SFT Training...")
trainer.train()
trainer.model.save_pretrained(sft_output_dir)
tokenizer.save_pretrained(sft_output_dir)
print("Stage 1 SFT Completed!")

## 5. Stage 2: Direct Preference Optimization (DPO) Alignment

In [ ]:
from trl import DPOTrainer

dpo_output_dir = "checkpoints/dpo_output"

dpo_train_ds = load_dataset("json", data_files="data/processed/dpo_train.json", split="train")
dpo_val_ds = load_dataset("json", data_files="data/processed/dpo_val.json", split="train")

dpo_args = TrainingArguments(
    output_dir=dpo_output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=50,
    save_steps=50,
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    beta=0.1,
    train_dataset=dpo_train_ds,
    eval_dataset=dpo_val_ds,
    tokenizer=tokenizer,
    args=dpo_args,
    max_length=2048,
    max_prompt_length=1024,
)

print("Starting DPO Alignment...")
dpo_trainer.train()
dpo_trainer.model.save_pretrained(dpo_output_dir)
tokenizer.save_pretrained(dpo_output_dir)
print("Stage 2 DPO Alignment Complete!")

## 6. Run Benchmark & Evaluation Suite

In [ ]:
import json

from src.evaluation.benchmark import BenchmarkRunner

runner = BenchmarkRunner(schemas_dir="data/schemas", output_dir="outputs/evaluation")
with open("data/processed/eval_test.json") as f:
    test_samples = json.load(f)

def eval_predict(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.0)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

report = runner.run_benchmark(eval_predict, test_samples, model_name="FunctionCraft-Qwen-1.5B-DPO")
print("Benchmark Results:")
print(json.dumps(report, indent=2))